# NGUYỄN THÀNH TRÍ - MSSV: 6451071079
## BÀI 4: PHÂN LỚP DỮ LIỆU IRIS BẰNG NEURAL NETWORK KẾT HỢP 10-FOLD CROSS VALIDATION
### (K-FOLD CROSS VALIDATION VỚI K = 10)

In [1]:
# Nguyễn Thành Trí - MSSV: 6451071079
print("Họ tên: Nguyễn Thành Trí - MSSV: 6451071079")

import os
import warnings
os.environ['TF_CPP_MIN_LOG_LEVEL'] = '3'
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
from sklearn.datasets import load_iris
from sklearn.model_selection import KFold
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, classification_report
import keras
from keras import layers, models

print("Đã nạp đầy đủ các thư viện cần thiết cho Bài 4.")

Họ tên: Nguyễn Thành Trí - MSSV: 6451071079


I0000 00:00:1794280312.024980   66234 cudart_stub.cc:31] Could not find cuda drivers on your machine, GPU will not be used.


I0000 00:00:1794280313.323119   66234 cudart_stub.cc:31] Could not find cuda drivers on your machine, GPU will not be used.


Đã nạp đầy đủ các thư viện cần thiết cho Bài 4.


### Câu a: Đọc bộ dữ liệu Iris từ sklearn
Hướng dẫn: Sử dụng hàm `load_iris()` từ thư viện `sklearn.datasets` để đọc dữ liệu.

In [2]:
# Nguyễn Thành Trí - MSSV: 6451071079
print("Họ tên: Nguyễn Thành Trí - MSSV: 6451071079")

iris = load_iris()
print("Đã nạp thành công bộ dữ liệu Iris từ sklearn.")
print(f"Số lượng mẫu: {iris.data.shape[0]}, Số thuộc tính: {iris.data.shape[1]}")
print(f"Danh sách các lớp: {iris.target_names.tolist()}")

Họ tên: Nguyễn Thành Trí - MSSV: 6451071079
Đã nạp thành công bộ dữ liệu Iris từ sklearn.
Số lượng mẫu: 150, Số thuộc tính: 4
Danh sách các lớp: ['setosa', 'versicolor', 'virginica']


### Câu b: Chuẩn hóa dữ liệu data về đoạn [0, 1]
Hướng dẫn: Gán X là data và Y là target. Chia mỗi giá trị của X cho giá trị lớn nhất trong cột tương ứng (dùng `X.max(axis=0)`).

In [3]:
# Nguyễn Thành Trí - MSSV: 6451071079
print("Họ tên: Nguyễn Thành Trí - MSSV: 6451071079")

X = iris.data.copy()
Y = iris.target.copy()

X_max = X.max(axis=0)
X = X / X_max

print("Đã chuẩn hóa ma trận thuộc tính X về đoạn [0, 1].")
print(f"Giá trị cực đại từng cột trước chuẩn hóa (X_max): {X_max}")
print(f"Giá trị nhỏ nhất sau chuẩn hóa: {X.min():.2f}")
print(f"Giá trị lớn nhất sau chuẩn hóa: {X.max():.2f}")

Họ tên: Nguyễn Thành Trí - MSSV: 6451071079
Đã chuẩn hóa ma trận thuộc tính X về đoạn [0, 1].
Giá trị cực đại từng cột trước chuẩn hóa (X_max): [7.9 4.4 6.9 2.5]
Giá trị nhỏ nhất sau chuẩn hóa: 0.04
Giá trị lớn nhất sau chuẩn hóa: 1.00


### Câu c: Định nghĩa hàm create_model khởi tạo và biên dịch mạng Neural Network
Kiến trúc mô hình (tương tự Bài 1.h):
- Hidden layer 1: 10 nơ-ron, hàm kích hoạt ReLU, Fully connected, input_shape=(4,).
- Hidden layer 2: 20 nơ-ron, hàm kích hoạt ReLU, Fully connected.
- Output layer: 3 nơ-ron, hàm kích hoạt Softmax, Fully connected.
- Biên dịch với: optimizer='adam', loss='sparse_categorical_crossentropy', metrics=['accuracy'].

In [4]:
# Nguyễn Thành Trí - MSSV: 6451071079
print("Họ tên: Nguyễn Thành Trí - MSSV: 6451071079")

def create_model():
    model = keras.Sequential([
        layers.Input(shape=(4,)),
        layers.Dense(10, activation='relu'),
        layers.Dense(20, activation='relu'),
        layers.Dense(3, activation='softmax')
    ], name="Iris_MLP_Classifier")
    model.compile(
        optimizer='adam',
        loss='sparse_categorical_crossentropy',
        metrics=['accuracy']
    )
    return model

# Khởi tạo mô hình mẫu và in tóm tắt kiến trúc
sample_model = create_model()
sample_model.summary()

Họ tên: Nguyễn Thành Trí - MSSV: 6451071079


Model: "Iris_MLP_Classifier"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ dense (Dense)                   │ (None, 10)             │            50 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 20)             │           220 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_2 (Dense)                 │ (None, 3)              │            63 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 333 (1.30 KB)

 Trainable params: 333 (1.30 KB)

 Non-trainable params: 0 (0.00 B)